In [ ]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 13.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing ins

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected -- go to Runtime > Change runtime type > T4 GPU, then re-run.")


CUDA available: True
GPU: Tesla T4


In [ ]:
from google.colab import files
import os

os.makedirs("data", exist_ok=True)
print("Select your train.jsonl file:")
uploaded = files.upload()

# Move whatever was uploaded into a fixed path, regardless of its original
# filename, so every cell below can assume one location.
uploaded_name = next(iter(uploaded))
TRAIN_FILE = "data/train.jsonl"
os.replace(uploaded_name, TRAIN_FILE)
print(f"Saved to {TRAIN_FILE}")

Select your train.jsonl file:


Saving train.jsonl to train.jsonl
Saved to data/train.jsonl


In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("qwen-uchtdorf-lora")
GGUF_OUTPUT_NAME = "qwen-uchtdorf-ft"  # becomes <name>.gguf

BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 2048
LORA_RANK = 16

ALPACA_TEMPLATE = """### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_RANK,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.7: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.10.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments


def format_example(example: dict) -> dict:
    text = ALPACA_TEMPLATE.format(**example) + tokenizer.eos_token
    return {"text": text}


dataset = load_dataset("json", data_files=TRAIN_FILE, split="train")
dataset = dataset.map(format_example)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        output_dir="checkpoints",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        seed=3407,
    ),
)

print(f"Training on {len(dataset)} examples...")
trainer.train()


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/101 [00:00<?, ? examples/s]

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/101 [00:00<?, ? examples/s]

Training on 101 examples...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 101 | Num Epochs = 3 | Total steps = 39
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
5,2.993497
10,2.373886
15,2.149553
20,1.960598
25,2.022990
30,1.873315
35,1.944653


Unsloth: Restored added_tokens_decoder metadata in checkpoints/checkpoint-39/tokenizer_config.json.


TrainOutput(global_step=39, training_loss=2.1535053008641953, metrics={'train_runtime': 77.7669, 'train_samples_per_second': 3.896, 'train_steps_per_second': 0.501, 'total_flos': 273803005335552.0, 'train_loss': 2.1535053008641953, 'epoch': 3.0})

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained_gguf(
    str(OUTPUT_DIR),
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"Wrote merged GGUF to {OUTPUT_DIR}/")
!ls -la {OUTPUT_DIR}


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in qwen-uchtdorf-lora/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:40<00:00, 40.71s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:55<00:00, 55.07s/it]


Unsloth: Merge process complete. Saved to `/content/qwen-uchtdorf-lora`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11734-mix-c208e5f (app-b11734-mix-c208e5f-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['qwen-uchtdorf-lora_gguf/Qwen2.5-1.5B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions co

In [ ]:
# Only if you want to test the fine-tuned model here in Colab by installing and
# running Ollama execute this block and others below

!apt-get install -y zstd -qq

In [ ]:
# Install Ollama inside this Colab VM (temporary -- gone when the session ends).
!curl -fsSL https://ollama.com/install.sh | sh

# Start the Ollama server in the background. The '&' is what makes this
# non-blocking -- without it, this cell would hang forever waiting for the
# server process to exit, instead of handing control back so the next cell
# can run.
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # give the server a moment to come up before we talk to it


In [ ]:
gguf_path = gguf_files[0]
modelfile_text = f'''FROM {gguf_path}

SYSTEM """Respond to the question or statement with a 2-5 sentence response in the voice of Elder Dieter F. Uchtdorf, the way he speaks in his talks."""

PARAMETER temperature 0.0
'''
Path("Modelfile").write_text(modelfile_text)

!ollama create qwen-uchtdorf-ft -f Modelfile


In [ ]:
# One test prompt, sent straight through the ollama CLI -- no LangChain
# needed for a single ad hoc check like this.
sample_input = ""

!ollama run qwen-uchtdorf-ft "{sample_input}"
